In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df = spark.read.table("sales_clean").toPandas()

# Create TotalSales
df['TotalSales'] = (df['Quantity'] * df['UnitPrice']) + df['TaxAmount']

# convert the date properly
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df = df.sort_values('OrderDate')

# Aggregate sales over time for prediction
daily_sales = df.groupby('OrderDate')['TotalSales'].sum().reset_index()


# this is our simple prediction model
from sklearn.linear_model import LinearRegression
import numpy as np

# Convert dates to numbers
daily_sales['Days'] = (daily_sales['OrderDate'] - daily_sales['OrderDate'].min()).dt.days

X = daily_sales[['Days']]
y = daily_sales['TotalSales']

model = LinearRegression()
model.fit(X, y)

# Predict next 30 days
future_days = np.arange(X.max()[0]+1, X.max()[0]+31).reshape(-1,1)
predictions = model.predict(future_days)


# Visualize our predictions
import matplotlib.pyplot as plt

plt.figure()
plt.plot(daily_sales['OrderDate'], y)
plt.plot(predictions)
plt.title("Sales Prediction")
plt.show()